In [ ]:
# ============================================
# JUSTICEGRAPH - GOOGLE COLAB VERSION
# FastAPI Backend with ngrok Tunnel
# ============================================

# =========================
# STEP 1: INSTALL PACKAGES
# =========================
print("Installing required packages...")
import subprocess
import sys

packages = [
    "fastapi", "uvicorn[standard]", "pandas", "numpy",
    "scikit-learn", "matplotlib", "seaborn", "pillow", "pyngrok"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ All packages installed successfully!\n")

# =========================
# IMPORTS
# =========================
from fastapi import FastAPI, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import base64
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
import re
import warnings
warnings.filterwarnings('ignore')

# For Colab server
from pyngrok import ngrok
import nest_asyncio
nest_asyncio.apply()

# =========================
# DATASET GENERATION
# =========================
print("Generating datasets...")

np.random.seed(42)

# State-City Mapping
state_city_map = {
    'Delhi': ['New Delhi', 'South Delhi', 'Rohini'],
    'Maharashtra': ['Mumbai', 'Pune', 'Nagpur'],
    'Karnataka': ['Bengaluru', 'Mysuru', 'Mangalore'],
    'Tamil Nadu': ['Chennai', 'Coimbatore', 'Madurai'],
    'West Bengal': ['Kolkata', 'Siliguri'],
    'Telangana': ['Hyderabad', 'Warangal'],
    'Gujarat': ['Ahmedabad', 'Surat', 'Vadodara'],
    'Rajasthan': ['Jaipur', 'Udaipur', 'Jodhpur'],
    'Uttar Pradesh': ['Lucknow', 'Noida', 'Kanpur'],
    'Punjab': ['Chandigarh', 'Amritsar', 'Ludhiana'],
    'Kerala': ['Kochi', 'Thiruvananthapuram', 'Kozhikode'],
    'Madhya Pradesh': ['Bhopal', 'Indore', 'Gwalior'],
    'Bihar': ['Patna', 'Gaya'],
    'Odisha': ['Bhubaneswar', 'Cuttack'],
    'Haryana': ['Gurgaon', 'Faridabad'],
    'Assam': ['Guwahati', 'Dibrugarh'],
    'Chhattisgarh': ['Raipur', 'Bilaspur'],
    'Jharkhand': ['Ranchi', 'Jamshedpur']
}

court_types = ['District Court', 'High Court', 'Sessions Court', 'Supreme Court']

# COURTS DATASET
courts_data = []
for i in range(120):
    court_id = f"COURT_{i+1:03d}"
    state = np.random.choice(list(state_city_map.keys()))
    city = np.random.choice(state_city_map[state])
    court_type = np.random.choice(court_types, p=[0.55, 0.25, 0.15, 0.05])

    judge_strength = {
        'District Court': np.random.randint(10, 40),
        'Sessions Court': np.random.randint(8, 30),
        'High Court': np.random.randint(50, 150),
        'Supreme Court': np.random.randint(200, 300)
    }[court_type]

    pending_cases = int(np.random.normal(8000, 2000)) if court_type != 'Supreme Court' else np.random.randint(20000, 50000)
    monthly_filing_rate = np.random.randint(100, 1000)
    monthly_disposal_rate = np.random.randint(80, monthly_filing_rate)
    avg_disposal_time_days = np.random.randint(180, 900)
    infrastructure_score = round(np.random.uniform(4.5, 9.5), 2)
    digitization_level = round(np.random.uniform(0.4, 0.98), 2)

    courts_data.append({
        'court_id': court_id,
        'court_name': f"{city} {court_type}",
        'city': city,
        'state': state,
        'court_type': court_type,
        'judge_strength': judge_strength,
        'pending_cases': pending_cases,
        'monthly_filing_rate': monthly_filing_rate,
        'monthly_disposal_rate': monthly_disposal_rate,
        'avg_disposal_time_days': avg_disposal_time_days,
        'infrastructure_score': infrastructure_score,
        'digitization_level': digitization_level
    })

df_courts = pd.DataFrame(courts_data)

# JUDGES DATASET
judge_specializations = [
    'Criminal', 'Civil', 'Constitutional', 'Corporate',
    'Family', 'Property', 'Tax', 'Cybercrime', 'Environmental'
]

def generate_judge_name(i):
    first_names = ["Amit", "Neha", "Ravi", "Sneha", "Arun", "Priya", "Sanjay",
                   "Deepa", "Manish", "Divya", "Rakesh", "Meena", "Rajesh", "Kavita"]
    last_names = ["Sharma", "Reddy", "Patel", "Iyer", "Singh", "Das", "Mehra",
                  "Menon", "Chatterjee", "Kumar", "Verma", "Rao", "Bose", "Pillai"]
    return f"Justice {np.random.choice(first_names)} {np.random.choice(last_names)}"

judges_data = []
for i in range(300):
    judge_id = f"JUDGE_{i+1:04d}"
    court = df_courts.sample(1).iloc[0]

    specialization = np.random.choice(judge_specializations, p=[0.2, 0.18, 0.1, 0.12, 0.15, 0.1, 0.05, 0.05, 0.05])
    experience = np.random.randint(5, 40)
    cases_handled = int(np.random.normal(1500, 600))
    avg_judgment_time = np.random.randint(45, 500)
    reversal_rate = round(np.random.uniform(0.02, 0.25), 3)
    bias_index = round(np.random.uniform(0.0, 0.45), 3)

    judges_data.append({
        'judge_id': judge_id,
        'judge_name': generate_judge_name(i),
        'court_id': court['court_id'],
        'court_name': court['court_name'],
        'state': court['state'],
        'specialization': specialization,
        'experience_years': experience,
        'cases_handled': cases_handled,
        'avg_judgment_time_days': avg_judgment_time,
        'reversal_rate': reversal_rate,
        'bias_index': bias_index,
        'rating_score': round(10 - (bias_index*5 + reversal_rate*10) + np.random.uniform(0, 2), 2)
    })

df_judges = pd.DataFrame(judges_data)

# CASES DATASET
case_types = [
    'Criminal', 'Civil', 'Constitutional', 'Corporate',
    'Family', 'Property', 'Tax', 'Cybercrime', 'Environmental', 'Labor'
]
case_statuses = ['Pending', 'Decided', 'Transferred', 'Dismissed', 'Adjourned']

def generate_case_title():
    petitioner = np.random.choice(["State of", "Union of India", "Mr.", "Ms.", "M/s", "People vs"])
    respondent = np.random.choice(["Ravi Kumar", "ABC Pvt Ltd", "Neha Sharma",
                                   "XYZ Corp", "Union Govt", "Sanjay Singh", "Amit Mehta"])
    return f"{petitioner} {respondent}"

cases_data = []
for i in range(7000):
    case_id = f"CASE_{i+1:06d}"
    court = df_courts.sample(1).iloc[0]
    judge_subset = df_judges[df_judges['court_id'] == court['court_id']]
    judge_id = np.random.choice(judge_subset['judge_id'].values) if len(judge_subset) > 0 else np.random.choice(df_judges['judge_id'].values)

    filing_date = datetime(2020, 1, 1) + timedelta(days=np.random.randint(0, 1600))
    case_type = np.random.choice(case_types, p=[0.22, 0.18, 0.05, 0.07, 0.1, 0.1, 0.08, 0.07, 0.05, 0.08])
    status = np.random.choice(case_statuses, p=[0.45, 0.3, 0.08, 0.1, 0.07])

    cases_data.append({
        'case_id': case_id,
        'case_title': generate_case_title(),
        'case_number': f"{np.random.randint(1, 9999)}/{np.random.randint(2020, 2025)}",
        'court_id': court['court_id'],
        'court_name': court['court_name'],
        'state': court['state'],
        'judge_id': judge_id,
        'case_type': case_type,
        'status': status,
        'filing_date': filing_date.strftime('%Y-%m-%d'),
        'hearing_count': np.random.randint(1, 25),
        'complexity_score': round(np.random.uniform(1.5, 9.8), 2),
        'case_value_lakhs': round(np.random.exponential(scale=80), 2),
        'days_pending': np.random.randint(5, 1800),
        'public_interest_tag': np.random.choice(['Yes', 'No'], p=[0.1, 0.9])
    })

df_cases = pd.DataFrame(cases_data)

# LAWS DATASET
laws_data = [
    {'law_id': 'IPC_302', 'law_name': 'IPC Section 302', 'category': 'Criminal',
     'description': 'Punishment for murder', 'severity': 10},
    {'law_id': 'IPC_420', 'law_name': 'IPC Section 420', 'category': 'Criminal',
     'description': 'Cheating and dishonestly inducing delivery of property', 'severity': 7},
    {'law_id': 'CPC_9', 'law_name': 'CPC Section 9', 'category': 'Civil',
     'description': 'Courts to try all civil suits unless barred', 'severity': 5},
    {'law_id': 'IT_66', 'law_name': 'IT Act Section 66', 'category': 'Cybercrime',
     'description': 'Computer-related offences', 'severity': 8},
    {'law_id': 'ART_21', 'law_name': 'Article 21', 'category': 'Constitutional',
     'description': 'Protection of life and personal liberty', 'severity': 10},
    {'law_id': 'ART_14', 'law_name': 'Article 14', 'category': 'Constitutional',
     'description': 'Equality before law', 'severity': 9},
    {'law_id': 'GST_73', 'law_name': 'GST Act Section 73', 'category': 'Tax',
     'description': 'Determination of tax not paid or short paid', 'severity': 6},
    {'law_id': 'ENV_15', 'law_name': 'Environment Act Section 15', 'category': 'Environmental',
     'description': 'Penalty for contravention of the provisions of the Act', 'severity': 8},
    {'law_id': 'LAB_25F', 'law_name': 'Industrial Disputes Act 25F', 'category': 'Labor',
     'description': 'Conditions precedent to retrenchment of workmen', 'severity': 6},
]

df_laws = pd.DataFrame(laws_data)

def map_law_to_case(case_type):
    subset = df_laws[df_laws['category'].str.lower() == case_type.lower()]
    if len(subset) > 0:
        return np.random.choice(subset['law_id'].values)
    else:
        return np.random.choice(df_laws['law_id'].values)

df_cases['law_id'] = df_cases['case_type'].apply(map_law_to_case)

# FEATURE ENGINEERING
df_courts['backlog_rate'] = (df_courts['monthly_filing_rate'] - df_courts['monthly_disposal_rate']) / df_courts['judge_strength']
df_courts['utilization_rate'] = df_courts['pending_cases'] / (df_courts['judge_strength'] * 100)
df_courts['efficiency_score'] = df_courts['monthly_disposal_rate'] / (df_courts['pending_cases'] + 1)
df_courts['risk_factors'] = (
    df_courts['backlog_rate'] * 0.3 +
    df_courts['utilization_rate'] * 0.3 +
    (1 - df_courts['efficiency_score']) * 0.2 +
    (1 - df_courts['digitization_level']) * 0.2
)

scaler = MinMaxScaler()
df_courts['backlog_risk_score'] = scaler.fit_transform(df_courts[['risk_factors']])
df_courts['risk_category'] = pd.cut(df_courts['backlog_risk_score'],
                                      bins=[0, 0.3, 0.6, 1.0],
                                      labels=['Low Risk', 'Moderate Risk', 'High Risk'])

print("✅ Datasets generated successfully!")
print(f"   - Courts: {len(df_courts)}")
print(f"   - Judges: {len(df_judges)}")
print(f"   - Cases: {len(df_cases)}")
print(f"   - Laws: {len(df_laws)}\n")

# =========================
# ML MODEL TRAINING
# =========================
print("Training ML models...")

# Model 1: Court Risk Prediction
X_features = ['judge_strength', 'pending_cases', 'monthly_filing_rate',
              'monthly_disposal_rate', 'avg_disposal_time_days',
              'infrastructure_score', 'digitization_level']

X = df_courts[X_features]
y = (df_courts['backlog_risk_score'] > 0.6).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# Model 2: Case Outcome Prediction
def generate_case_text(row):
    return f"The {row['case_type']} case with complexity {row['complexity_score']} and {row['hearing_count']} hearings"

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df_cases['case_text'] = df_cases.apply(generate_case_text, axis=1)
df_cases['outcome'] = df_cases['status'].apply(lambda x: 1 if x == 'Decided' else 0)
df_cases['clean_text'] = df_cases['case_text'].apply(clean_text)

X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    df_cases['clean_text'], df_cases['outcome'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=500)
X_train_vec = vectorizer.fit_transform(X_text_train)
X_test_vec = vectorizer.transform(X_text_test)

lr_model = LogisticRegression(max_iter=500, random_state=42)
lr_model.fit(X_train_vec, y_text_train)

print("✅ ML models trained successfully!\n")

# =========================
# FASTAPI APPLICATION
# =========================

app = FastAPI(
    title="JusticeGraph API",
    description="AI-Powered Legal Analytics Platform",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ROOT
@app.get("/")
def root():
    return {
        "message": "JusticeGraph API is active",
        "version": "1.0.0",
        "status": "running",
        "endpoints": ["/courts", "/judges", "/cases", "/laws", "/summary", "/analyze"]
    }

# COURTS
@app.get("/courts")
def get_courts(state: str = None, limit: int = 100):
    df = df_courts.copy()
    if state:
        df = df[df['state'].str.contains(state, case=False, na=False)]
    df = df.head(limit)
    df['risk_category'] = df['risk_category'].astype(str)
    return {"total": len(df), "data": df.to_dict(orient='records')}

# JUDGES
@app.get("/judges")
def get_judges(state: str = None, specialization: str = None, limit: int = 100):
    df = df_judges.copy()
    if state:
        df = df[df['state'].str.contains(state, case=False, na=False)]
    if specialization:
        df = df[df['specialization'].str.contains(specialization, case=False, na=False)]
    return {"total": len(df.head(limit)), "data": df.head(limit).to_dict(orient='records')}

# CASES
@app.get("/cases")
def get_cases(state: str = None, case_type: str = None, status: str = None, limit: int = 100):
    df = df_cases.copy()
    if state:
        df = df[df['state'].str.contains(state, case=False, na=False)]
    if case_type:
        df = df[df['case_type'].str.contains(case_type, case=False, na=False)]
    if status:
        df = df[df['status'].str.contains(status, case=False, na=False)]
    return {"total": len(df.head(limit)), "data": df.head(limit).to_dict(orient='records')}

# LAWS
@app.get("/laws")
def get_laws():
    return {"total": len(df_laws), "data": df_laws.to_dict(orient='records')}

# SUMMARY
@app.get("/summary")
def get_summary():
    return {
        "total_courts": len(df_courts),
        "total_judges": len(df_judges),
        "total_cases": len(df_cases),
        "pending_cases": len(df_cases[df_cases['status'] == 'Pending']),
        "decided_cases": len(df_cases[df_cases['status'] == 'Decided']),
        "high_risk_courts": len(df_courts[df_courts['backlog_risk_score'] > 0.6]),
        "avg_disposal_time": round(df_courts['avg_disposal_time_days'].mean(), 2)
    }

# ANALYZE
@app.post("/analyze")
def analyze_case(
    state: str = Query(...),
    case_type: str = Query(...),
    hearing_count: int = Query(5),
    complexity_score: float = Query(5.0)
):
    # Find court
    matching_courts = df_courts[df_courts['state'].str.contains(state, case=False, na=False)]
    if len(matching_courts) == 0:
        matching_courts = df_courts
    user_court = matching_courts.iloc[0]

    # Predict outcome
    case_desc = f"The {case_type} case with complexity {complexity_score} and {hearing_count} hearings"
    case_clean = clean_text(case_desc)
    case_vec = vectorizer.transform([case_clean])
    prediction = lr_model.predict(case_vec)[0]
    prob = lr_model.predict_proba(case_vec)[0]

    return {
        "court": {
            "name": user_court['court_name'],
            "state": user_court['state'],
            "risk_score": float(user_court['backlog_risk_score']),
            "pending_cases": int(user_court['pending_cases'])
        },
        "prediction": {
            "outcome": "Favorable" if prediction == 1 else "Unfavorable",
            "confidence": round(float(max(prob)) * 100, 2),
            "favorable_probability": round(float(prob[1]) * 100, 2)
        },
        "timeline": {
            "expected_days": int(user_court['avg_disposal_time_days'])
        }
    }

# VISUALS
@app.get("/visuals")
def get_visuals(chart_type: str = "risk_distribution"):
    plt.figure(figsize=(10, 6))

    if chart_type == "risk_distribution":
        risk_counts = df_courts['risk_category'].value_counts()
        plt.pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%',
               colors=['#27AE60', '#F39C12', '#E74C3C'])
        plt.title('Court Risk Distribution')
    else:
        status_counts = df_cases['status'].value_counts()
        plt.bar(status_counts.index, status_counts.values, color='#3498DB')
        plt.title('Case Status Distribution')
        plt.xticks(rotation=45)

    buffer = io.BytesIO()
    plt.savefig(buffer, format='png', dpi=150, bbox_inches='tight')
    buffer.seek(0)
    image_base64 = base64.b64encode(buffer.read()).decode()
    plt.close()

    return {"chart_type": chart_type, "image_base64": image_base64}

# =========================
# START SERVER WITH NGROK
# =========================

print("="*80)
print("STARTING JUSTICEGRAPH API SERVER (COLAB VERSION)")
print("="*80)

# Set your ngrok auth token (GET FREE TOKEN FROM: https://ngrok.com/)
# Uncomment and add your token:
ngrok.set_auth_token("34ggMiJtBIcDOLDmTbLrJAXi7wc_71tNCumvVFoF6XYrQDQqC")

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"\n✅ PUBLIC URL: {public_url}")
print(f"   Use this URL to access your API from anywhere!")
print(f"\n📚 API Documentation: {public_url}/docs")
print(f"📖 Alternative Docs: {public_url}/redoc")
print("\n" + "="*80)
print("Server is running... Press Ctrl+C to stop")
print("="*80 + "\n")

# Run server
import nest_asyncio
import uvicorn
nest_asyncio.apply()

await uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000)).serve()


Installing required packages...
✅ All packages installed successfully!

Generating datasets...
✅ Datasets generated successfully!
   - Courts: 120
   - Judges: 300
   - Cases: 7000
   - Laws: 9

Training ML models...
✅ ML models trained successfully!

STARTING JUSTICEGRAPH API SERVER (COLAB VERSION)

✅ PUBLIC URL: NgrokTunnel: "https://unsectionalised-clingiest-dorris.ngrok-free.dev" -> "http://localhost:8000"
   Use this URL to access your API from anywhere!

📚 API Documentation: NgrokTunnel: "https://unsectionalised-clingiest-dorris.ngrok-free.dev" -> "http://localhost:8000"/docs
📖 Alternative Docs: NgrokTunnel: "https://unsectionalised-clingiest-dorris.ngrok-free.dev" -> "http://localhost:8000"/redoc

Server is running... Press Ctrl+C to stop



INFO:     Started server process [558]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2409:40f2:39:e584:739d:f41a:110c:1ea8:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40f2:39:e584:739d:f41a:110c:1ea8:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2409:40f2:39:e584:9c8e:26ff:fe04:aa6:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40f2:39:e584:9c8e:26ff:fe04:aa6:0 - "GET / HTTP/1.1" 200 OK
